# 02 — The Autonomy Envelope

**Chapter 2 teaching lab — §2.2 + Figure 2.1**

> **Teaching outcome:** Build a graduated autonomy envelope.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from harnesslab.envelope import Envelope, envelope_violations
from harnesslab.viz import envelope_view
from harnesslab.playground import envelope_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(2)

## Concept
"Autonomous" is not a boolean. The **autonomy envelope** states, in testable terms, how far an agent
may go on its own along several independent axes:

| Axis | Question |
|---|---|
| **level** | observe → recommend → draft → stage → execute reversible → execute consequential |
| **blast radius** | how many people, systems, or records can one action touch? |
| **external reach** | do effects leave the organization? |
| **duration** | how long does authority last before it must be renewed? |

An action is inside the envelope only if it is inside on **every** axis.

## Why this matters
A global `autonomous=True` flag grants everything or nothing, so teams either block useful work or
silently allow consequential work. Separate axes let you grant "draft anything, stage small changes,
never contact customers" and *test* that statement.

## Diagram (Figure 2.1)
```mermaid
flowchart LR
  A[Proposed action] --> L["level ≤ STAGE"] & B["blast radius ≤ 2"] & X["no external reach"] & D["within 1 hour"]
  L & B & X & D --> V{all four hold?}
  V -->|yes| IN([inside the envelope])
  V -->|no| OUT([outside: report every violated axis])
  classDef ok fill:#16a34a22,stroke:#16a34a
  classDef bad fill:#dc262622,stroke:#dc2626
  class IN ok
  class OUT bad
```

## Read the implementation

In [ ]:
show_source(Envelope, envelope_violations)

## Experiment

In [ ]:
env = Envelope(AutonomyLevel.STAGE, max_blast_radius=2, allow_external=False, max_duration_seconds=3600)
catalog = [
    Action("search",  AutonomyLevel.OBSERVE, Reversibility.READ_ONLY, 1),
    Action("draft",   AutonomyLevel.DRAFT,   Reversibility.REVERSIBLE, 1),
    Action("stage",   AutonomyLevel.STAGE,   Reversibility.REVERSIBLE, 2),
    Action("publish", AutonomyLevel.EXECUTE_CONSEQUENTIAL, Reversibility.IRREVERSIBLE, 8, external=True),
]
envelope_view(catalog, env, envelope_violations)

## Expected behavior

In [ ]:
expected = {"search": [], "draft": [], "stage": [], "publish": ["level", "blast_radius", "external"]}
assert {a.name: envelope_violations(a, env) for a in catalog} == expected

# Authority decays with time even for an action that was fine a minute ago.
assert envelope_violations(catalog[0], env, elapsed_seconds=4000) == ["duration"]
envelope_view(catalog[:1], env, envelope_violations, elapsed_seconds=4000)

## Red-team round (§2.2)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense checks only the level axis.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.2"), run_gauntlet(NAIVE, "2.2")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.2")

## Playground
Move the sliders and find a *low-level* action that is still outside the envelope.

In [ ]:
envelope_playground(env)

## Challenge
Define an envelope for a **coding workflow** that can inspect, edit locally, run tests, and stage a
pull request — but cannot deploy. Write the `Action` catalog and assertions proving `deploy_production`
is outside on at least two axes while `run_tests` is inside on all of them.